# Standard Benchmarks — Evaluation Results

| Model | Abbreviation |
|---|---|
| SuperPoint + SuperGlue | SP+SG |
| LoFTR | LoFTR |
| SuperPoint + LightGlue | SP+LG |
| SuperPoint + LSD + GlueStick | SP+GS |

**Vocabulary**

| Metric | Benchmarks | Description |
|---|---|---|
| `gt_match_precision@3px` | HP, MD, SN | Fraction of predicted matches agreeing with GT (homography/depth-based, 3 px) |
| `gt_match_recall@3px` | HP, MD, SN | Fraction of all GT-matchable pairs actually found |
| `prec@{1,3,5}px` | HP | Match precision at symmetric homography reprojection thresholds |
| `reproj_prec@{1,3,5}px` | MD, SN | Match precision at depth-based reprojection thresholds |
| `num_matches` / `num_keypoints` | All | Average match/keypoint volume per pair |

In [ ]:
import json
from pathlib import Path
BASE = Path.cwd()
RESULTS_FOLDER = BASE / 'outputs/results'  # Expecting same structure as in Glue Factory
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Plot style
sns.set_theme(style = 'whitegrid', font_scale = 1.15)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

MODEL_DIRS = {
    'superpoint+superglue_eval':     'SP+SG',
    'loftr_eval':                    'LoFTR',
    'superpoint+lightglue_eval':     'SP+LG',
    'superpoint+lsd+gluestick_eval': 'SP+GS',
}
MODEL_ORDER = ['SP+SG', 'LoFTR', 'SP+LG', 'SP+GS']
PALETTE = dict(zip(MODEL_ORDER, sns.color_palette('Set2', 4)))
BENCHMARKS = ['megadepth1500', 'scannet1500', 'hpatches']
BENCH_LABELS = {'megadepth1500': 'MegaDepth-1500', 'scannet1500': 'ScanNet-1500', 'hpatches': 'HPatches'}

# Load summaries
records = []
for bench in BENCHMARKS:
    for model_dir, model_name in MODEL_DIRS.items():
        p = RESULTS_FOLDER / bench / model_dir / 'summaries.json'
        if p.exists():
            with open(p) as f:
                d = json.load(f)
            d['benchmark'] = bench
            d['model'] = model_name
            records.append(d)
df_summary = pd.DataFrame(records).set_index(['benchmark', 'model'])

# Load per-pair metrics from results.h5
pair_rows = []
for bench in BENCHMARKS:
    for model_dir, model_name in MODEL_DIRS.items():
        rp = RESULTS_FOLDER / bench / model_dir / 'results.h5'
        if not rp.exists():
            continue
        with h5py.File(str(rp), 'r') as f:
            n_pairs = None
            datasets = {}
            for k in f.keys():
                obj = f[k]
                if isinstance(obj, h5py.Dataset):
                    arr = obj[:]
                    datasets[k] = arr
                    if n_pairs is None and obj.dtype.kind in ('f', 'i', 'u'):
                        n_pairs = len(arr)
            if n_pairs is None:
                continue
            for i in range(n_pairs):
                row = {'benchmark': bench, 'model': model_name, 'pair_idx': i}
                for k, arr in datasets.items():
                    if arr.dtype.kind in ('f', 'i', 'u'):
                        row[k] = float(arr[i])
                    elif arr.dtype.kind in ('S', 'O', 'U'):
                        val = arr[i]
                        row[k] = val.decode() if isinstance(val, bytes) else str(val)
                pair_rows.append(row)

df = pd.DataFrame(pair_rows)
print(f'Loaded {len(records)} summary sets, {len(df)} per-pair records')

---
## 1. Summary Tables

In [ ]:
def display_summary(bench, cols):
    sub = df.loc[df['benchmark'] == bench, cols].copy().astype(float)
    sub_group = sub.groupby(df['model']).mean().round(4)
    display(sub_group)
    

# MegaDepth-1500
print('\n── MegaDepth-1500 ───────────────────────────────')
md_cols = ['reproj_prec@1px','reproj_prec@3px','reproj_prec@5px',
           'num_matches','num_keypoints']
display_summary('megadepth1500', md_cols)

# ScanNet-1500
print('\n── ScanNet-1500 ─────────────────────────────────')
sn_cols = ['reproj_prec@1px','reproj_prec@3px','reproj_prec@5px',
           'num_matches','num_keypoints']
display_summary('scannet1500', sn_cols)

# HPatches
print('── HPatches ─────────────────────────────────────')
hp_cols = ['prec@1px','prec@3px','prec@5px',
           'num_matches','num_keypoints']
display_summary('hpatches', hp_cols)

---
## 2. Precision vs Recall Scatter

Each point is one model. F1 iso-lines show the precision–recall trade-off.

In [ ]:
fig, axes = plt.subplots(1, len(BENCHMARKS), figsize = (6 * len(BENCHMARKS), 5))

labels = ['(a)', '(b)', '(c)']

for ax, bench in zip(axes, BENCHMARKS):
    rec  = df.loc[df['benchmark'] == bench].groupby('model')['gt_match_recall@3px'].mean()
    prec = df.loc[df['benchmark'] == bench].groupby('model')['gt_match_precision@3px'].mean()
    
    for model in MODEL_ORDER:
        if model in rec.index:
            ax.scatter(rec.loc[model], prec.loc[model], s = 140,
                       color = PALETTE[model], label = model, zorder = 5, edgecolors = 'k', linewidth = 0.5)
            ax.annotate(model, (rec.loc[model], prec.loc[model]),
                        textcoords = 'offset points', xytext = (5, 5), fontsize = 10)

    # F1 iso-lines
    r_range = np.linspace(0.01, 1.0, 200)
    for f1 in [0.3, 0.5, 0.7, 0.9]:
        p_range = f1 * r_range / (2 * r_range - f1)
        mask = (p_range >= 0) & (p_range <= 1)
        ax.plot(r_range[mask], p_range[mask], 'k--', lw = 0.5, alpha = 0.25)
        ax.annotate(f'F1={f1}', xy=(r_range[mask][-1], p_range[mask][-1]),
                    fontsize = 8, color = 'gray')

    ax.set_xlim(0, 1.05)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('GT Match Recall')
    ax.set_ylabel('GT Match Precision')
    ax.set_title(f'{labels[BENCHMARKS.index(bench)]} {BENCH_LABELS[bench]}', fontweight = 'bold', fontsize = 24)
    if bench == BENCHMARKS[0]:
        ax.legend(fontsize = 10, frameon = False)
    
plt.tight_layout()
plt.show()